# RideOps support: diagnosis
Every number in the write-up comes from this notebook. Data: `data/rideops_support_tickets.csv`, pulled 30 Sep 2026.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
pd.set_option('display.width', 200)
from src.load import load_tickets, run_checks
from src import metrics as m, diagnosis as dg, backtest as bt
from src.rules import DEFAULT_THRESHOLDS
df = load_tickets()
run_checks(df)

{'ticket_id_unique': True,
 'response_after_created': True,
 'first_close_after_created': True,
 'final_close_after_first_close': True,
 'closed_has_final_close': True,
 'open_has_no_final_close': True}

## Golden counts (section 4.2 'done when')

In [2]:
h = m.headline(df)
{k: h[k] for k in ['tickets','escalations','fixed','backlog_30_sep']}

{'tickets': 1444, 'escalations': 486, 'fixed': 211, 'backlog_30_sep': 48}

## Problem summary

In [3]:
h

{'tickets': 1444,
 'backlog_31_jul': 30,
 'backlog_30_sep': 48,
 'escalations': 486,
 'fixed': 211,
 'wasted': 275,
 'wasted_pct': 0.565843621399177,
 'handoffs_total': 777,
 'handoffs_wasted': 528,
 'med_ttr_non_esc': 12.566666666666666,
 'med_ttr_esc': 86.775,
 'open_hours_share_esc': 0.7142249142892392,
 'reopen_pct': 0.10457063711911357,
 'csat_mean': 3.7037037037037037,
 'csat_resp_rate': 0.4487534626038781}

In [4]:
dg.monthly_volume(df)

month
2026-07    390
2026-08    462
2026-09    592
dtype: int64

## D1 · Volume grew in every category

In [5]:
dg.d1_volume_by_category(df)

month,category,2026-07,2026-08,2026-09,growth_jul_sep
0,Employee address / geocode change,13,23,39,2.000000
1,Cab delay / no-show complaint,29,46,72,1.482759
2,Billing / invoice discrepancy,24,22,42,0.750000
3,How-to / feature request,23,43,38,0.652174
4,Report export failing,57,59,90,0.578947
5,Login / access issue,29,31,40,0.379310
6,OTP / boarding issue,28,30,37,0.321429
7,Driver / vehicle compliance docs,18,16,23,0.277778
8,Live tracking / GPS incorrect,65,80,83,0.276923
9,Roster / booking not syncing,104,112,128,0.230769


## D2 · Escalation is what makes tickets slow

In [6]:
dg.d2_ttr_by_handoffs(df)

,eng_handoffs,count,median_ttr_h
0,0,944,12.566667
1,1,245,58.650000
2,2,153,110.366667
3,3,54,169.908333


## D3 · Most escalations shouldn't have gone to engineering

In [7]:
dg.d3_outcomes(df)

,eng_outcome,tickets,handoffs
0,Fixed,211,249
1,Insufficient info - returned to L1,89,207
2,Config issue - returned to L1,84,201
3,Duplicate of known issue,56,68
4,Not a bug - how-to,24,28
5,Not reproducible,22,24


## D4 · Wasted escalations are the slowest tickets

In [8]:
dg.d4_ttr_by_outcome(df)

,eng_outcome,median_ttr_h,mean_handoffs,tickets
0,Config issue - returned to L1,128.566667,2.394737,76
1,Insufficient info - returned to L1,121.116667,2.303797,79
2,Not a bug - how-to,68.666667,1.173913,23
3,Duplicate of known issue,67.491667,1.203704,54
4,Fixed,64.833333,1.175879,199
5,Not reproducible,62.300000,1.095238,21


## D5 · Duplicates of known issues are growing

In [9]:
by_month, by_tpl = dg.d5_duplicates(df)
display(by_month)
by_tpl.head(10)

month
2026-07     8
2026-08    21
2026-09    27
dtype: int64

,category,template,escalations,duplicates,dup_share
0,Report export failing,Scheduled report email not received,15,9,0.600000
1,Report export failing,Monthly trip report download stuck at loading,24,9,0.375000
2,Report export failing,Exported report missing N trips,17,7,0.411765
3,Roster / booking not syncing,Cancelled bookings still appearing in tomorrow...,37,5,0.135135
4,Report export failing,Excel export of billing report is blank,12,4,0.333333
5,Roster / booking not syncing,Booking cutoff changed but roster still uses o...,34,4,0.117647
6,Roster / booking not syncing,Ad-hoc booking done by admin not reflecting fo...,31,3,0.096774
7,Roster / booking not syncing,Roster uploaded via Excel but N employees missing,25,3,0.120000
8,Live tracking / GPS incorrect,Cab location frozen on map for last N mins,20,2,0.100000
9,Live tracking / GPS incorrect,Employee app shows cab N km away while cab is ...,25,2,0.080000


## D6 · Escalation depends on the agent, not the ticket

In [10]:
agents, prio = dg.d6_agents_priority(df)
display(agents)
prio

,assigned_agent,volume,esc_pct,wasted_pct
2,Karan B,217,0.161290,0.628571
3,Neha P,171,0.286550,0.612245
7,Vikram S,176,0.301136,0.396226
1,Arjun K,178,0.325843,0.568966
4,Priya D,186,0.327957,0.540984
6,Sneha M,176,0.346591,0.491803
0,Ananya R,189,0.402116,0.473684
5,Rahul T,151,0.615894,0.752688


,priority,esc_pct
0,P1,0.328358
1,P2,0.348485
2,P3,0.331536
3,P4,0.339623


## D7 · Fast closing is partly fake speed

In [11]:
dg.d7_fast_close(df)

,assigned_agent,med_first_close_non_esc_h,reopen_pct_non_esc,reopen_pct,csat
2,Karan B,6.425000,0.280220,0.294931,3.132653
3,Neha P,11.416667,0.057377,0.081871,3.936709
5,Rahul T,11.600000,0.068966,0.079470,3.523810
1,Arjun K,11.616667,0.066667,0.078652,3.853333
0,Ananya R,11.683333,0.053097,0.068783,3.837209
7,Vikram S,12.733333,0.040650,0.045455,3.918605
4,Priya D,13.333333,0.072000,0.091398,3.783133
6,Sneha M,14.933333,0.052174,0.051136,3.717949


## D8 · Clients feel the handoffs and reopens

In [12]:
d8 = dg.d8_client_feel(df)
for k, v in d8.items():
    print(k); display(v.round(2))

csat_by_handoffs


eng_handoffs
0    4.01
1    3.56
2    2.80
3    2.10
Name: csat_score, dtype: float64

csat_reopened


reopened
False    3.84
True     2.35
Name: csat_score, dtype: float64

ttr_reopened


reopened
False    18.52
True     70.41
Name: ttr_h, dtype: float64

## Success-measure baselines

In [13]:
p = df.groupby('priority')['frt_h'].median().round(2)
{'reopen_pct': round(h['reopen_pct'],3), 'csat_mean': round(h['csat_mean'],2), 'csat_resp_rate': round(h['csat_resp_rate'],3), 'P1_frt_h': p['P1'], 'P2_frt_h': p['P2']}

{'reopen_pct': 0.105,
 'csat_mean': 3.7,
 'csat_resp_rate': 0.449,
 'P1_frt_h': 0.28,
 'P2_frt_h': 1.05}

In [14]:
m.weekly_kpis(df).round(3)

C:\Users\Hemant Kumar Rohilla\AppData\Local\Temp\ipykernel_19236\1555954635.py:1: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  m.weekly_kpis(df).round(3)


,week,volume,backlog,med_ttr,esc_pct,wasted_pct,reopen_pct,csat,csat_resp_rate,p1p2_frt
0,2026-06-29,40,11,7.883,0.325,0.385,0.125,3.562,0.400,0.433
1,2026-07-06,87,12,17.358,0.310,0.444,0.126,3.727,0.379,0.333
2,2026-07-13,87,14,13.667,0.310,0.556,0.069,3.860,0.494,0.750
3,2026-07-20,97,24,24.083,0.454,0.659,0.052,3.683,0.423,0.833
4,2026-07-27,88,17,33.800,0.443,0.538,0.170,3.436,0.443,1.550
5,2026-08-03,107,20,15.225,0.271,0.586,0.112,3.860,0.467,0.567
6,2026-08-10,103,24,30.617,0.417,0.628,0.087,3.627,0.495,0.733
7,2026-08-17,117,27,22.567,0.299,0.543,0.103,3.945,0.470,0.650
8,2026-08-24,114,21,21.592,0.307,0.457,0.061,3.660,0.465,0.500
9,2026-08-31,105,38,22.225,0.371,0.667,0.152,3.458,0.457,0.767


## Backtest (learn Jul–Aug, test September)

In [15]:
th, tune_res = bt.tune(df)
_, test = bt.split(df)
table = bt.stats_table(df, test)
for name, t in [('tuned', th), ('plan defaults', DEFAULT_THRESHOLDS)]:
    r = bt.run_backtest(df, t, table)
    print(name, t)
    print({k: round(v, 3) if isinstance(v, float) else v for k, v in r['confusion'].items()})
    print('baseline precision', round(r['baseline_precision'], 3), '| claim supported:', r['claim_supported'],
          '| handoffs avoided', r['handoffs_avoided'], '| proxy hours', round(r['proxy_hours_saved']))
    display(r['by_action'])

tuned Thresholds(dup_rate=0.2, config_howto_rate=0.4, info_rate=1.01, agent_wasted_rate=0.65, min_live=0, use_recent_dup=False, recent_days=30, min_n=5)
{'TP': 45, 'FP': 30, 'FN': 66, 'TN': 48, 'flagged': 75, 'n': 189, 'catch_rate': 0.405, 'precision': 0.6, 'precision_lb': 0.505, 'held_back_fixed': 30, 'f05': 0.547}
baseline precision 0.587 | claim supported: True | handoffs avoided 75 | proxy hours 1693


,action,flagged,wasted,fixed_held_back
0,Link to known issue,33,21,12
1,Try resolving at L1 first,12,4,8
2,Escalate only with the checklist completed,30,20,10
3,Escalate,114,66,0


plan defaults Thresholds(dup_rate=0.3, config_howto_rate=0.35, info_rate=0.25, agent_wasted_rate=0.65, min_live=1, use_recent_dup=True, recent_days=30, min_n=5)
{'TP': 95, 'FP': 69, 'FN': 16, 'TN': 9, 'flagged': 164, 'n': 189, 'catch_rate': 0.856, 'precision': 0.579, 'precision_lb': 0.515, 'held_back_fixed': 69, 'f05': 0.619}
baseline precision 0.587 | claim supported: False | handoffs avoided 180 | proxy hours 3574


,action,flagged,wasted,fixed_held_back
0,Link to known issue,134,80,54
1,Try resolving at L1 first,11,6,5
2,Escalate only with the checklist completed,19,9,10
3,Escalate,25,16,0


In [16]:
r = bt.run_backtest(df, th, table)
{k: r[k] for k in ['p_value_vs_baseline', 'dup_base_rate', 'link_flagged', 'link_duplicates',
                   'link_dup_precision', 'link_dup_lift', 'link_dup_p_value', 'handoffs_avoided',
                   'proxy_hours_saved']}

{'p_value_vs_baseline': 0.46049444515052923,
 'dup_base_rate': 0.14285714285714285,
 'link_flagged': 33,
 'link_duplicates': 16,
 'link_dup_precision': 0.48484848484848486,
 'link_dup_lift': 3.393939393939394,
 'link_dup_p_value': 3.0528953527550944e-06,
 'handoffs_avoided': 75,
 'proxy_hours_saved': 1693.1250000000007}

In [17]:
r['by_trigger']

,trigger,flagged,wasted,duplicates,precision
0,agent,30,20,1,0.666667
1,config,12,4,0,0.333333
2,dup,33,21,16,0.636364


In [18]:
r['baselines']

,strategy,flagged,TP,catch_rate,precision,held_back_fixed
0,Flag nothing,0,0,0.0,NaN,0
1,Flag everything,189,111,1.0,0.587302,78


## Walk-forward validation (tune on prior months, test the next)

In [19]:
wf = bt.walk_forward(df)
display(wf)
wf.attrs

,test_month,escalations,baseline_precision,flagged,TP,FP,catch_rate,precision,held_back_fixed,beats_baseline,thresholds
0,Aug 2026,151,0.562914,77,48,29,0.564706,0.623377,29,True,"{'dup_rate': 0.3, 'config_howto_rate': 0.4, 'i..."
1,Sep 2026,189,0.587302,75,45,30,0.405405,0.600000,30,True,"{'dup_rate': 0.2, 'config_howto_rate': 0.4, 'i..."


{'pooled_precision': 0.6118421052631579, 'pooled_baseline': 0.5764705882352941}

In [20]:
bt.threshold_grid(df, table)

,dup_rate,agent_wasted_rate,config_howto_rate,info_rate,min_live,use_recent_dup,flagged,TP,FP,catch_rate,precision,held_back_fixed
0,0.25,0.60,1.01,1.01,0,True,111,66,45,0.594595,0.594595,45
1,0.25,0.60,1.01,1.01,0,False,71,43,28,0.387387,0.605634,28
2,0.25,0.65,1.01,1.01,0,True,104,63,41,0.567568,0.605769,41
3,0.25,0.65,1.01,1.01,0,False,58,37,21,0.333333,0.637931,21
4,0.25,1.01,1.01,1.01,0,True,84,51,33,0.459459,0.607143,33
5,0.25,1.01,1.01,1.01,0,False,27,17,10,0.153153,0.629630,10
6,0.30,0.60,1.01,1.01,0,True,111,66,45,0.594595,0.594595,45
7,0.30,0.60,1.01,1.01,0,False,69,41,28,0.369369,0.594203,28
8,0.30,0.65,1.01,1.01,0,True,104,63,41,0.567568,0.605769,41
9,0.30,0.65,1.01,1.01,0,False,54,33,21,0.297297,0.611111,21
